In [1]:
pip install shared_utils

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importing necessary package 
import pandas as pd 
import geopandas as gpd
import numpy as np
import google.auth
import os
import gcsfs
from calitp_data_analysis.sql import get_engine
from calitp_data_analysis import utils
db_engine = get_engine()
credentials, project = google.auth.default()
from pandas.tseries.holiday import USFederalHolidayCalendar
fs = gcsfs.GCSFileSystem()

pd.set_option('display.max_columns', None)

In [3]:
GCS_FILE_PATH  = 'gs://calitp-analytics-data/data-analyses/ahsc_grant/ahsc_riderships'

In [4]:
bart_ridership = pd.read_excel(f"{GCS_FILE_PATH}/bart_ridership.xlsx")
big_blue_bus_ridership = pd.read_excel(f"{GCS_FILE_PATH}/big_blue_bus_ridership.xlsx")
caltrain_ridership = pd.read_excel(f"{GCS_FILE_PATH}/caltrain_ridership.xlsx")
# culver_citybus_ridership = pd.read_excel(f"{GCS_FILE_PATH}/culver_citybus_ridership.xlsx")
foothill_transit_ridership = pd.read_excel(f"{GCS_FILE_PATH}/foothill_transit_ridership.xlsx")
fresno_area_express_ridership = pd.read_excel(f"{GCS_FILE_PATH}/fresno_area_express_ridership.xlsx")
gold_coast_transit_ridership = pd.read_excel(f"{GCS_FILE_PATH}/gold_coast_transit_ridership.xlsx")
golden_gate_park_shuttle_ridership = pd.read_excel(f"{GCS_FILE_PATH}/golden_gate_park_shuttle_ridership.xlsx")
golden_gate_transit_ridership = pd.read_excel(f"{GCS_FILE_PATH}/golden_gate_transit_ridership.xlsx")
long_beach_transit_ridership=  pd.read_excel(f"{GCS_FILE_PATH}/long_beach_transit_ridership.xlsx")
# octa_ridership = pd.read_excel(f"{GCS_FILE_PATH}/octa_ridership.xlsx")
# omni_trans_ridership= pd.read_excel(f"{GCS_FILE_PATH}/omni_trans_ridership.xlsx")
riverside_transit_ridership= pd.read_excel(f"{GCS_FILE_PATH}/riverside_transit_ridership.xlsx")
# sacrt_bus_ridership = pd.read_excel(f"{GCS_FILE_PATH}/sacrt_bus_ridership.xlsx")
samtrans_ridership = pd.read_excel(f"{GCS_FILE_PATH}/samtrans_ridership.xlsx")
# santa_cruz_metro_ridership = pd.read_excel(f"{GCS_FILE_PATH}/santa_cruz_metro_ridership.xlsx")
# sbmtd_ridership = pd.read_excel(f"{GCS_FILE_PATH}/sbmtd_ridership.xlsx")
sdmts_ridership = pd.read_excel(f"{GCS_FILE_PATH}/sdmts_ridership.xlsx")
# sunline_transit_ridership = pd.read_excel(f"{GCS_FILE_PATH}/sunline_transit_ridership.xlsx")


In [5]:
def add_day_type(df, date_col, output_col='day_type'):
    """
    Classify each row as 'Weekday', 'Saturday', or 'Sunday' based on a specified date column.
    This is useful when daily datasets include labels such as 'Holiday' or when no day_type
    column exists. The function converts the date column, identifies the day of week, and
    standardizes the day-type classification accordingly. Either a start_date or end_date
    column can be used for datasets with daily reporting.
    """
    df = df.copy()

    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')

    dow = df[date_col].dt.dayofweek  

    df[output_col] = np.select(
        condlist=[dow.eq(5), dow.eq(6)],
        choicelist=['Saturday', 'Sunday'],
        default='Weekday'
    )

    return df

In [6]:
# Counting number of days without holidays
def count_service_days(row):
    dates = pd.date_range(row.start_date, row.end_date)

    if row.day_type == "Weekday":
        return sum(d.weekday() < 5 for d in dates)
    elif row.day_type == "Saturday":
        return sum(d.weekday() == 5 for d in dates)
    elif row.day_type == "Sunday":
        return sum(d.weekday() == 6 for d in dates)

In [17]:
# Counting number of days with holidays
def count_service_days_wo_holidays(row):
    dates = pd.date_range(row.start_date, row.end_date)

    if row.day_type == "Weekday":
        return sum((d.weekday() < 5) and (d not in holiday_set) for d in dates)

    elif row.day_type == "Saturday":
        return sum((d.weekday() == 5) and (d not in holiday_set) for d in dates)

    elif row.day_type == "Sunday":
        return sum((d.weekday() == 6) and (d not in holiday_set) for d in dates)

In [69]:
def compute_averages(df, ridership_cols, group_cols):
    """
    Computes average of one or more ridership columns per group in long format.

    Parameters:
    - df: pandas DataFrame
    - ridership_cols: list of columns to average (e.g., ['daily_boardings', 'daily_alightings'])
    - group_cols: list of columns to group by (e.g., ['stop_name', 'day_type'])

    Returns:
    - DataFrame with columns: group_cols + average of each ridership column
    """
    
    if isinstance(ridership_cols, str):
        ridership_cols = [ridership_cols]  # allow single string input
    
    averaged = df.groupby(group_cols)[ridership_cols].mean().reset_index()
    
    # rename each column to indicate it's averaged
    averaged = averaged.rename(columns={col: f"average_{col}" for col in ridership_cols})
    
    return averaged

In [71]:
bart_ridership = add_day_type(bart_ridership, date_col='start_date')
bart_ridership['daily_boardings'] = pd.to_numeric(bart_ridership['daily_boardings'], errors='coerce')
average_bart_ridership = compute_averages(
    df=bart_ridership,
    ridership_cols=["daily_boardings", "daily_alightings"],
    group_cols=["dataset_id", "organization_name", "service_name", 
                "stop_id", "stop_name", "stop_lat", "stop_lon", "day_type"]
)

average_bart_ridership.sample(5)

,dataset_id,organization_name,service_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings
75,011CF30F49575609,San Francisco Bay Area Rapid Transit District,Bay Area Rapid Transit,903409,Walnut Creek,37.905791,-122.067327,Saturday,1660.519231,1674.115385
64,011CF30F49575609,San Francisco Bay Area Rapid Transit District,Bay Area Rapid Transit,902909,Fremont,37.557480,-121.976619,Sunday,810.750000,819.692308
50,011CF30F49575609,San Francisco Bay Area Rapid Transit District,Bay Area Rapid Transit,902409,San Leandro,37.721784,-122.160740,Weekday,3255.329502,3318.099617
142,011CF30F49575609,San Francisco Bay Area Rapid Transit District,Bay Area Rapid Transit,909209,Warm Springs / South Fremont,37.502285,-121.939395,Sunday,425.538462,436.519231
135,011CF30F49575609,San Francisco Bay Area Rapid Transit District,Bay Area Rapid Transit,908209,Pittsburg Center,38.016847,-121.889062,Saturday,281.134615,309.326923


In [8]:
big_blue_bus_ridership.columns = big_blue_bus_ridership.columns.str.lower()
big_blue_bus_ridership['service_day'] = big_blue_bus_ridership['service_day'].str.strip().str.capitalize()


big_blue_bus_ridership = big_blue_bus_ridership.rename(columns={
    'route_number': 'route_short_name',
    'service_day': 'day_type',
    'average_daily_boardings':'average_daily_boardings_period',
    'average_daily_alightings':'average_daily_alightings_period'    
})


# Sum boardings/alightings across directions within same stop & period
total_big_blue_bus_ridership = (
    big_blue_bus_ridership
    .groupby(['day_type','route_short_name','route_name',
              'stop_id','stop_name','stop_lat','stop_lon',
              'start_date','end_date'])
    .agg({
        'average_daily_boardings_period':'sum',
        'average_daily_alightings_period':'sum'
    })
    .reset_index()
)


In [9]:
big_blue_bus_ridership['start_date'] = pd.to_datetime(big_blue_bus_ridership['start_date'])
big_blue_bus_ridership['end_date'] = pd.to_datetime(big_blue_bus_ridership['end_date'])

total_big_blue_bus_ridership["n_days"] = total_big_blue_bus_ridership.apply(count_service_days, axis=1)

# Taking weighted average of the ridership across 4 different time periods
averaged_total_big_blue_bus_ridership = (
    total_big_blue_bus_ridership
    .groupby(['day_type','route_short_name','route_name',
              'stop_id','stop_lat','stop_lon'])
    .apply(lambda x: pd.Series({
        'avg_daily_boardings': np.average(x['average_daily_boardings_period'], weights=x['n_days']),
        'avg_daily_alightings': np.average(x['average_daily_alightings_period'], weights=x['n_days'])
    }))
    .reset_index()
)


# Set the ridership basis flag
averaged_total_big_blue_bus_ridership["daily_ridership_basis"] = "reported_avg_daily"

averaged_total_big_blue_bus_ridership.sample(5)

/tmp/ipykernel_930/2421701129.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series({


,day_type,route_short_name,route_name,stop_id,stop_lat,stop_lon,avg_daily_boardings,avg_daily_alightings,daily_ridership_basis
2237,Weekday,8,Ocean Park Blvd & Westwood Bl/UCLA,2957,34.009271,-118.492574,5.660871,0.416635,reported_avg_daily
718,Saturday,18,UCLA - Abbot Kinney,2083,34.054587,-118.455742,2.723356,2.806829,reported_avg_daily
2024,Weekday,7,Pico Blvd,1106,34.010970,-118.488919,29.880715,1.843366,reported_avg_daily
556,Saturday,14,Bundy Dr & Centinela Ave,2456,34.033867,-118.455318,10.768792,10.147075,reported_avg_daily
2067,Weekday,7,Pico Blvd,2177,34.049772,-118.410856,17.535932,13.656621,reported_avg_daily


In [38]:
caltrain_ridership = caltrain_ridership.rename(columns={
    'Origin Station': 'stop_name',
    'Date Type': 'day_type',
    'Average Ridership':'average_daily_boardings_period',  
})

caltrain_ridership = caltrain_ridership[
    caltrain_ridership["day_type"] != "Holiday"
]

# Getting number of days and using US Federal Holidays
start_date_caltrain = '2023-11-01'
end_date_caltrain = '2025-07-31'

#Getting all the federal holidays between the start and end date
cal = USFederalHolidayCalendar()
holidays = cal.holidays(start=start_date_caltrain, end=end_date_caltrain)

# Setting holidays to calculate the number of weekdays, saturday and sunday without holidays 
holiday_set = set(holidays)

caltrain_ridership["start_date"] = pd.to_datetime(caltrain_ridership["start_date"])
caltrain_ridership["end_date"] = pd.to_datetime(caltrain_ridership["end_date"])

caltrain_ridership["n_days"] = caltrain_ridership.apply(count_service_days_wo_holidays, axis=1)

In [39]:
# Taking weighted average of the ridership across 20 months time periods
averaged_caltrain_ridership = (
    caltrain_ridership
    .groupby(['day_type','stop_name'])
    .apply(lambda x: pd.Series({
        'avg_daily_boardings': np.average(x['average_daily_boardings_period'], weights=x['n_days'])
    }))
    .reset_index()
)

# Set the ridership basis flag
averaged_caltrain_ridership["daily_ridership_basis"] = "reported_avg_daily"

averaged_caltrain_ridership.sample(5)

/tmp/ipykernel_930/1127038250.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series({


,day_type,stop_name,avg_daily_boardings,daily_ridership_basis
85,Weekday,San Mateo,1074.189854,reported_avg_daily
86,Weekday,Santa Clara,718.467622,reported_avg_daily
87,Weekday,South San Francisco,559.501648,reported_avg_daily
88,Weekday,Sunnyvale,1408.195136,reported_avg_daily
89,Weekday,Tamien,198.883066,reported_avg_daily


In [74]:
# Add day_type from the date column
foothill_transit_ridership = add_day_type(foothill_transit_ridership, date_col="date")

# Grouping columns
group_cols = [
    "date", "route_short_name", "stop_code", "stop_lat", "stop_lon",
    "start_date", "end_date", "day_type"
]

# Sum boardings and alightings, with explicit output names
total_ridership_foothill = (
    foothill_transit_ridership
    .groupby(group_cols, as_index=False)
    .agg(
        daily_boardings=("boardings", "sum"),
        daily_alightings=("alightings", "sum")
    )
)

# Set the ridership basis flag
total_ridership_foothill["daily_ridership_basis"] = "reported_daily"

# Calculate average ridership per day type and stop
total_ridership_foothill['daily_boardings'] = pd.to_numeric(total_ridership_foothill['daily_boardings'], errors='coerce')
average_foothill_ridership = compute_averages(
    df=total_ridership_foothill,
    ridership_cols=["daily_boardings", "daily_alightings"],
    group_cols=["route_short_name", "stop_code", 
                "stop_lat", "stop_lon", "day_type"]
)

average_foothill_ridership.head(5)


,route_short_name,stop_code,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings
0,178,23,34.034964,-117.919263,Saturday,1.689655,0.310345
1,178,23,34.034964,-117.919263,Sunday,1.750000,0.281250
2,178,23,34.034964,-117.919263,Weekday,2.144186,0.218605
3,178,24,34.035320,-117.919706,Saturday,2.341463,0.097561
4,178,24,34.035320,-117.919706,Sunday,2.375000,0.156250


In [76]:
# Add day_type from the date column
fresno_area_express_ridership = add_day_type(fresno_area_express_ridership, date_col="Date")

fresno_area_express_ridership["daily_ridership_basis"] = "reported_daily"


fresno_area_express_ridership = fresno_area_express_ridership.rename(columns={
    'StopID': 'stop_id',
    'StopLabel': 'stop_name',
    'ProjectedBoarding': 'daily_boardings',
    'ProjectedAlighting': 'daily_alightings'
})

# Calculate average ridership per day type and stop
fresno_area_express_ridership['daily_boardings'] = pd.to_numeric(fresno_area_express_ridership['daily_boardings'], errors='coerce')
average_fresno_area_express_ridership = compute_averages(
    df=fresno_area_express_ridership,
    ridership_cols=["daily_boardings", "daily_alightings"],
    group_cols=["stop_id",  "stop_name", "day_type"]
)

average_fresno_area_express_ridership.head(5)

,stop_id,stop_name,day_type,average_daily_boardings,average_daily_alightings
0,5,NE BRAWLEY - SHIELDS,Saturday,35.329309,25.519283
1,5,NE BRAWLEY - SHIELDS,Sunday,33.548114,25.662840
2,5,NE BRAWLEY - SHIELDS,Weekday,51.142125,29.915370
3,6,SE SHAW - BRAWLEY,Saturday,7.004770,4.812710
4,6,SE SHAW - BRAWLEY,Sunday,5.616939,3.298467


In [77]:
# Add day_type from the date column
golden_gate_park_shuttle_ridership = add_day_type(golden_gate_park_shuttle_ridership, date_col="Date")

group_cols = [
    "Date", "day_type", "Stop", "start_date", "end_date"]

# Sum AVG On 
total_golden_gate_park_shuttle_ridership = (golden_gate_park_shuttle_ridership.groupby(
    group_cols, as_index=False).agg(
        daily_boardings=("Ridership", "sum"),
    )
)

total_golden_gate_park_shuttle_ridership["daily_ridership_basis"] = "reported_daily"

total_golden_gate_park_shuttle_ridership = total_golden_gate_park_shuttle_ridership.rename(columns={
    'Stop': 'stop_name',
    'Date': 'date'
})

# Calculate average ridership per day type and stop
total_golden_gate_park_shuttle_ridership['daily_boardings'] = pd.to_numeric(total_golden_gate_park_shuttle_ridership['daily_boardings'], errors='coerce')
average_golden_gate_park_shuttle_ridership = compute_averages(
    df=total_golden_gate_park_shuttle_ridership,
    ridership_cols=["daily_boardings"],
    group_cols=["stop_name", "day_type"]
)

average_golden_gate_park_shuttle_ridership.head(5)

,stop_name,day_type,average_daily_boardings
0,10th Ave/ De Young EB,Saturday,9.211538
1,10th Ave/ De Young EB,Sunday,7.076923
2,10th Ave/ De Young EB,Weekday,5.862069
3,10th Ave/ De Young WB,Saturday,9.519231
4,10th Ave/ De Young WB,Sunday,8.442308


In [79]:
# Add day_type from the date column
golden_gate_transit_ridership = add_day_type(golden_gate_transit_ridership, date_col="date")

golden_gate_transit_ridership = golden_gate_transit_ridership.rename(columns={
    'STOP_NUMBER': 'stop_id',
    'STOP_NAME': 'stop_name',
    'ROUTE': 'route_short_name',
})

# Remove everything in parentheses (including the parentheses) at the end of the string
golden_gate_transit_ridership['stop_name'] = golden_gate_transit_ridership['stop_name'].str.replace(r'\s*\(.*\)$', '', regex=True)

group_cols = [
    "day_type", "route_short_name", "stop_id", "stop_name", "end_date", "start_date"]

# Sum AVG On and AVG Off
total_golden_gate_transit_ridership = (golden_gate_transit_ridership.groupby(
    group_cols, as_index=False).agg(
        daily_boardings=("BOARDINGS", "sum"),
        daily_alightings=("ALIGHTINGS", "sum")
    )
)

total_golden_gate_transit_ridership["daily_ridership_basis"] = "reported_daily"

# Calculate average ridership per day type and stop
total_golden_gate_transit_ridership['daily_boardings'] = pd.to_numeric(total_golden_gate_transit_ridership['daily_boardings'], errors='coerce')
average_golden_gate_transit_ridership = compute_averages(
    df=total_golden_gate_transit_ridership,
    ridership_cols=["daily_boardings", "daily_alightings"],
    group_cols=["route_short_name", "stop_id", "stop_name", "day_type"]
)



average_golden_gate_transit_ridership.head(5)

,route_short_name,stop_id,stop_name,day_type,average_daily_boardings,average_daily_alightings
826,150,40030,Van Ness Ave & Clay St,Weekday,14.363636,7.090909
72,101,40698,DeLong Ave & Reichert Ave,Saturday,5.000000,1.000000
938,150,42245,Van Ness Ave & Eddy St,Saturday,1.500000,12.750000
178,101,41228,E Washington St & Vallejo St,Saturday,1.500000,4.250000
378,114,40143,Miller Ave & Una Way,Weekday,5.857143,0.000000


In [80]:
long_beach_transit_ridership["daily_ridership_basis"] = "reported_avg_daily"

group_cols = [
    "DayType", "Route", "StopID", "StopName", "end_date", "start_date", "daily_ridership_basis"]

# Sum AVG On and AVG Off
total_long_beach_transit_ridership = (long_beach_transit_ridership.groupby(
    group_cols, as_index=False).agg(
        daily_boardings=("Boardings", "sum"),
        daily_alightings=("Alightings", "sum")
    )
)

total_long_beach_transit_ridership = total_long_beach_transit_ridership.rename(columns={
    'StopID': 'stop_id',
    'StopName': 'stop_name',
    'DayType': 'day_type',
    'Route': 'route_short_name',
})


total_long_beach_transit_ridership.head(5)

,route_short_name,stop_id,stop_name,day_type,average_daily_boardings,average_daily_alightings
0,1,2001,2727 Del Amo Blvd N,Saturday,2.582828,0.000000
1,1,2001,2727 Del Amo Blvd N,Sunday,4.114699,0.209604
2,1,2001,2727 Del Amo Blvd N,Weekday,4.760665,7.940551
3,1,2002,2660 Del Amo Blvd S,Saturday,0.000000,0.000000
4,1,2002,2660 Del Amo Blvd S,Sunday,0.000000,0.000000


In [11]:
riverside_transit_ridership = add_day_type(riverside_transit_ridership, date_col='date')

group_cols = [
    "date", "Stop ID", "Route", "Longitude", "Latitude", "start_date", 
    "end_date", "day_type"]

# Sum AVG On and AVG Off
total_riverside_transit_ridership = (riverside_transit_ridership.groupby(
    group_cols, as_index=False).agg(
    daily_boardings = ('ridership', 'sum'))
)

total_riverside_transit_ridership = total_riverside_transit_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Route': 'route_short_name',
    'Latitude': 'stop_lat',
    'Longitude': 'stop_lon'

})




total_riverside_transit_ridership["daily_ridership_basis"] = "reported_daily"
total_riverside_transit_ridership.sample(5)

,date,stop_id,route_short_name,stop_lon,stop_lat,start_date,end_date,day_type,daily_boardings,daily_ridership_basis
49862,2025-01-30,3406,79,-117.139912,33.552444,2025-01-30,2025-01-30,Weekday,1,reported_daily
177645,2025-04-13,1029,1,-117.438848,33.927260,2025-04-13,2025-04-13,Sunday,10,reported_daily
394967,2025-09-24,1996,11,-117.243712,33.931872,2025-09-24,2025-09-24,Weekday,1,reported_daily
268031,2025-07-12,1794,15,-117.460664,33.902808,2025-07-12,2025-07-12,Saturday,1,reported_daily
269707,2025-07-13,1932,19,-117.226456,33.935324,2025-07-13,2025-07-13,Sunday,10,reported_daily


In [12]:
samtrans_ridership = add_day_type(samtrans_ridership, date_col='APC Date')

samtrans_ridership = samtrans_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Route': 'route_short_name',
    'Lat': 'stop_lat',
    'Lon': 'stop_lon',
    'APC Date': 'date',
    'Ons': 'daily_boardings',
    'Offs': 'daily_alightings',
})

samtrans_ridership["daily_ridership_basis"] = "reported_daily"
samtrans_ridership.sample(5)

,route_short_name,date,stop_id,Stop Name,daily_boardings,daily_alightings,stop_lat,stop_lon,day_type,start_date,end_date,daily_ridership_basis
55945,ECR,2025-08-22,341158,El Camino Real & State St,18,35,37.574062,-122.340286,Weekday,2025-08-22,2025-08-22,reported_daily
8296,120,2025-08-29,332314,Southgate Ave & Lincoln Ave,80,41,37.679637,-122.486793,Weekday,2025-08-29,2025-08-29,reported_daily
35083,295,2025-08-08,341620,256 37th Ave-San Mateo Medical Cent,1,0,37.531851,-122.300792,Weekday,2025-08-08,2025-08-08,reported_daily
19448,142,2025-08-25,335630,San Bruno BART-Bay 7 Outer Busway,41,40,37.641709,-122.198972,Weekday,2025-08-25,2025-08-25,reported_daily
33687,292,2025-08-30,331605,Bayshore Blvd & Fitzgerald Ave,2,31,37.727436,-122.401626,Saturday,2025-08-30,2025-08-30,reported_daily


In [13]:
group_cols = [
    "Route", "Route Name", "Stop ID",  "Stop Name", "day_type", "start_date", 
    "end_date"]

# Sum AVG On and AVG Off
total_sdmts_ridership = (sdmts_ridership.groupby(
    group_cols, as_index=False).agg(
        daily_boardings=("Average On", "sum"),
        daily_alightings=("Average Off", "sum")
    )
)

total_sdmts_ridership["Route Name"] = total_sdmts_ridership["Route Name"].str.split(":", n=1).str[1]

total_sdmts_ridership = total_sdmts_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Route': 'route_short_name',
    'Route Name': 'route_name',
    'Stop Name': 'stop_name',
})

total_sdmts_ridership["daily_ridership_basis"] = "reported_avg_daily"

total_sdmts_ridership.head(5)

,route_short_name,route_name,stop_id,stop_name,day_type,start_date,end_date,daily_boardings,daily_alightings,daily_ridership_basis
0,1,Fashion Valley-La Mesa,10106,University Av & 10th Av,Saturday,2024-09-01,2025-01-25,16.513085,8.709722,reported_avg_daily
1,1,Fashion Valley-La Mesa,10106,University Av & 10th Av,Sunday,2024-09-01,2025-01-25,14.264056,8.324885,reported_avg_daily
2,1,Fashion Valley-La Mesa,10106,University Av & 10th Av,Weekday,2024-09-01,2025-01-25,26.228026,15.629832,reported_avg_daily
3,1,Fashion Valley-La Mesa,10111,University Av & Vermont St,Saturday,2024-09-01,2025-01-25,42.220322,14.795541,reported_avg_daily
4,1,Fashion Valley-La Mesa,10111,University Av & Vermont St,Sunday,2024-09-01,2025-01-25,29.279929,10.192920,reported_avg_daily
5,1,Fashion Valley-La Mesa,10111,University Av & Vermont St,Weekday,2024-09-01,2025-01-25,59.583915,17.940893,reported_avg_daily
6,1,Fashion Valley-La Mesa,10114,University Av & Richmond St,Saturday,2024-09-01,2025-01-25,9.976243,6.169591,reported_avg_daily
7,1,Fashion Valley-La Mesa,10114,University Av & Richmond St,Sunday,2024-09-01,2025-01-25,7.180691,8.281819,reported_avg_daily
8,1,Fashion Valley-La Mesa,10114,University Av & Richmond St,Weekday,2024-09-01,2025-01-25,14.963778,14.173884,reported_avg_daily
9,1,Fashion Valley-La Mesa,10129,El Cajon Bl & Alabama St,Saturday,2024-09-01,2025-01-25,7.576505,5.638085,reported_avg_daily
